# PepDesign-Active: ConvVAE Training on Google Colab
**Purpose:** Train the peptide-specific ConvVAE on 52,517 food-derived peptides using Colab's free T4 GPU.
**Expected runtime:** ~10 minutes with GPU.
**Output:** VAE model weights + latent vectors + training loss curve → downloaded to your computer.

In [ ]:
# @title 1. Setup: Enable GPU + Install dependencies
!pip install -q pandas numpy scikit-learn torch torchvision

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import time, os, json, random
from google.colab import files

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# @title 2. Upload peptide library CSV
print("Please upload merged_peptide_library.csv from your computer.")
print("File location on your PC: D:\\projects\\walnut-peptide-pilot\\data\\merged_peptide_library.csv")
uploaded = files.upload()

import io
csv_name = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[csv_name]))
unique = df.drop_duplicates(subset=['peptide'])
unique = unique[unique['length'] >= 2]
peptides = unique['peptide'].tolist()

# Use 52,517 peptides (matching paper)
if len(peptides) > 52517:
    random.seed(42)
    peptides = random.sample(peptides, 52517)

print(f'Loaded {len(peptides)} unique peptides')
print(f'Length: mean={np.mean([len(p) for p in peptides]):.1f}, min={min(len(p) for p in peptides)}, max={max(len(p) for p in peptides)})')
print(f'Sample: {peptides[:5]}')

In [ ]:
# @title 3. Data preparation: one-hot encoding
AA_LIST = list('ACDEFGHIKLMNPQRSTVWY')
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_LIST)}
MAX_LEN = 20
N_AA = 20

def peptide_to_onehot(seq):
    mat = np.zeros((MAX_LEN, N_AA), dtype=np.float32)
    for i, aa in enumerate(seq[:MAX_LEN]):
        if aa in AA_TO_IDX:
            mat[i, AA_TO_IDX[aa]] = 1.0
    return mat

print('One-hot encoding (this takes ~30 seconds on Colab)...')
t0 = time.time()
onehot = np.stack([peptide_to_onehot(p) for p in peptides])
onehot = np.transpose(onehot, (0, 2, 1))  # (N, 20, 20)
print(f'Done in {time.time()-t0:.1f}s. Shape: {onehot.shape}')

In [ ]:
# @title 4. ConvVAE Model Definition
class ConvVAE(nn.Module):
    """Conv-VAE for peptide sequences (exactly as described in manuscript Section 2.2)."""
    def __init__(self, latent_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        # Encoder: 20 -> 32 -> 64 -> pool(5) -> FC(128) -> (mu, logvar)
        self.enc_conv1 = nn.Conv1d(N_AA, 32, 3, padding=1)
        self.enc_conv2 = nn.Conv1d(32, 64, 3, padding=1)
        self.enc_bn1 = nn.BatchNorm1d(32)
        self.enc_bn2 = nn.BatchNorm1d(64)
        self.enc_pool = nn.AdaptiveAvgPool1d(5)
        self.enc_fc1 = nn.Linear(64 * 5, 128)
        self.enc_fc_mu = nn.Linear(128, latent_dim)
        self.enc_fc_logvar = nn.Linear(128, latent_dim)
        # Decoder
        self.dec_fc1 = nn.Linear(latent_dim, 128)
        self.dec_fc2 = nn.Linear(128, 64 * 5)
        self.dec_deconv1 = nn.ConvTranspose1d(64, 32, 3, padding=1)
        self.dec_bn1 = nn.BatchNorm1d(32)
        self.dec_deconv2 = nn.ConvTranspose1d(32, N_AA, 3, padding=1)

    def encode(self, x):
        h = F.relu(self.enc_bn1(self.enc_conv1(x)))
        h = F.relu(self.enc_bn2(self.enc_conv2(h)))
        h = self.enc_pool(h)
        h = h.view(h.size(0), -1)
        h = F.relu(self.enc_fc1(h))
        return self.enc_fc_mu(h), self.enc_fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.dec_fc1(z))
        h = F.relu(self.dec_fc2(h))
        h = h.view(h.size(0), 64, 5)
        h = F.relu(self.dec_bn1(self.dec_deconv1(h)))
        h = self.dec_deconv2(h)
        return h

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

    def decode_latent(self, z_np):
        """Decode numpy latent vector to peptide sequence."""
        self.eval()
        with torch.no_grad():
            if isinstance(z_np, np.ndarray):
                z = torch.tensor(z_np, dtype=torch.float32).unsqueeze(0).to(device)
            logits = self.decode(z)
            indices = logits.squeeze(0).argmax(dim=0).cpu().numpy()
            return ''.join(AA_LIST[i] for i in indices if i < len(AA_LIST))

print('Model defined.')
model = ConvVAE(latent_dim=64).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# @title 5. Train ConvVAE (80 epochs, ~5 min on T4 GPU)
def vae_loss(recon_x, x, mu, logvar, beta=0.3):
    """Beta-VAE loss: reconstruction CE + KL divergence."""
    B, C, L = x.shape
    target = x.argmax(dim=1)  # (B, L)
    mask = (x.sum(dim=1) > 0).float()  # padding mask
    recon = recon_x.permute(0, 2, 1).reshape(-1, C)
    target = target.reshape(-1)
    mask = mask.reshape(-1)
    ce = F.cross_entropy(recon, target, reduction='none')
    bce = (ce * mask).sum() / (mask.sum() + 1e-8)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / B
    return bce + beta * kl

X = torch.tensor(onehot, dtype=torch.float32)
dataset = TensorDataset(X)
loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f'Training on {device}...')
print(f'{len(peptides)} peptides, {len(loader)} batches/epoch, 80 epochs')

losses = []
t_start = time.time()

for epoch in range(80):
    model.train()
    beta = min(0.3, 0.3 * (epoch + 1) / 40)  # linear anneal over first 40 epochs
    epoch_loss = 0.0
    for (batch,) in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        recon, mu, logvar = model(batch)
        loss = vae_loss(recon, batch, mu, logvar, beta)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(loader)
    losses.append(avg_loss)
    if (epoch + 1) % 10 == 0:
        elapsed = time.time() - t_start
        print(f'  Epoch {epoch+1:3d}/80 | loss={avg_loss:.4f} | beta={beta:.3f} | {elapsed:.0f}s elapsed')

total_time = time.time() - t_start
print(f'\nTraining complete in {total_time:.1f}s ({total_time/60:.1f} min)')
print(f'Initial loss: {losses[0]:.2f}, Final loss: {losses[-1]:.2f}')

In [ ]:
# @title 6. Evaluate: reconstruction accuracy
model.eval()
test_n = min(5000, len(peptides))
test_onehot = onehot[:test_n]
test_tensor = torch.tensor(test_onehot, dtype=torch.float32).to(device)

with torch.no_grad():
    recon_test, _, _ = model(test_tensor)
    recon_indices = recon_test.argmax(dim=1).cpu().numpy()  # (N, 20)

identities = []
for i in range(test_n):
    orig_seq = peptides[i]
    recon_seq = ''.join(AA_LIST[j] for j in recon_indices[i] if j < len(AA_LIST))
    min_len = min(len(orig_seq), len(recon_seq))
    if min_len > 0:
        matches = sum(1 for k in range(min_len) if orig_seq[k] == recon_seq[k])
        identities.append(matches / max(len(orig_seq), 1))
    else:
        identities.append(0.0)

mean_id = np.mean(identities)
print(f'Mean reconstruction accuracy: {mean_id:.3f} ({mean_id*100:.1f}%)')
print(f'Median: {np.median(identities):.3f}, Min: {np.min(identities):.3f}, Max: {np.max(identities):.3f}')

In [ ]:
# @title 7. Extract latent vectors for all peptides
print('Extracting latent vectors...')
model.eval()
latents = []
batch_size = 1024
with torch.no_grad():
    for i in range(0, len(onehot), batch_size):
        batch = torch.tensor(onehot[i:i+batch_size], dtype=torch.float32).to(device)
        mu, _ = model.encode(batch)
        latents.append(mu.cpu().numpy())
latents = np.vstack(latents).astype(np.float32)
print(f'Latent vectors: {latents.shape} (N, 64)')

# PCA explained variance of latent space
from sklearn.decomposition import PCA
pca = PCA(n_components=min(64, latents.shape[1]))
pca.fit(latents)
var_ratio = pca.explained_variance_ratio_.sum()
print(f'PCA explained variance of VAE latent space: {var_ratio:.3f} ({var_ratio*100:.1f}%)')

In [ ]:
# @title 8. Generate training loss figure (Supplementary Fig S1)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(losses)+1), losses, 'b-', linewidth=1.5)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('VAE Loss', fontsize=12)
ax.set_title('ConvVAE Training Loss (Supplementary Fig. S1)', fontsize=13, fontweight='bold')
ax.axvline(x=40, color='gray', linestyle='--', alpha=0.5, label='End of beta annealing')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_s1_vae_loss.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Initial loss: {losses[0]:.2f}')
print(f'Final loss:   {losses[-1]:.2f}')

In [ ]:
# @title 9. Generate PCA projection figure (Supplementary Fig S2)
from sklearn.decomposition import PCA as PCA_vis

# Use RPES scores if available, otherwise random coloring
np.random.seed(42)
vis_n = min(3000, len(latents))
vis_idx = np.random.choice(len(latents), vis_n, replace=False)
vis_lat = latents[vis_idx]

pca_vis = PCA_vis(n_components=2)
lat_2d = pca_vis.fit_transform(vis_lat)

# Color by hydrophobicity as proxy for bioactivity signal
KD_SCALE = {'A':1.8,'C':2.5,'D':-3.5,'E':-3.5,'F':2.8,'G':-0.4,'H':-3.2,'I':4.5,'K':-3.9,'L':3.8,'M':1.9,'N':-3.5,'P':-1.6,'Q':-3.5,'R':-4.5,'S':-0.8,'T':-0.7,'V':4.2,'W':-0.9,'Y':-1.3}
colors = []
for idx in vis_idx:
    seq = peptides[idx]
    hydro = np.mean([KD_SCALE.get(aa, 0.0) for aa in seq])
    colors.append(hydro)

fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(lat_2d[:, 0], lat_2d[:, 1], c=colors, cmap='RdYlBu', s=5, alpha=0.6)
plt.colorbar(sc, ax=ax, label='Mean Hydrophobicity')
ax.set_xlabel(f'PC1 ({pca_vis.explained_variance_ratio_[0]*100:.1f}%)', fontsize=12)
ax.set_ylabel(f'PC2 ({pca_vis.explained_variance_ratio_[1]*100:.1f}%)', fontsize=12)
ax.set_title('VAE Latent Space PCA Projection (Supplementary Fig. S2)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_s2_latent_pca.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# @title 10. Save everything and download

# Save model weights
torch.save(model.state_dict(), 'conv_vae_weights.pt')

# Save latent vectors
np.savez_compressed('vae_latents.npz', latents=latents, peptides=np.array(peptides))

# Save loss curve
pd.DataFrame({'epoch': range(1, len(losses)+1), 'loss': losses}).to_csv('vae_loss.csv', index=False)

# Save summary
summary = {
    'model': 'ConvVAE',
    'latent_dim': 64,
    'n_peptides': len(peptides),
    'epochs': 80,
    'initial_loss': float(losses[0]),
    'final_loss': float(losses[-1]),
    'reconstruction_accuracy': float(mean_id),
    'latent_pca_variance': float(var_ratio),
    'training_time_seconds': float(total_time),
    'device': str(device),
}
with open('training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('=' * 50)
print('TRAINING COMPLETE - Summary:')
print('=' * 50)
for k, v in summary.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

print('\nDownloading files...')
files.download('conv_vae_weights.pt')
files.download('vae_latents.npz')
files.download('vae_loss.csv')
files.download('training_summary.json')
files.download('fig_s1_vae_loss.png')
files.download('fig_s2_latent_pca.png')

## Next Steps After Colab

After downloading all 6 files, place them in:
```
D:\projects\sleep-deprivation-project\pepdesign\models\
```

Then run the local pipeline:
```bash
python pepdesign/phase1_pipeline.py  # Will use VAE latents instead of PCA
```

This will re-run the full experiment suite with the REAL ConvVAE latent space.